# prestructured_train

> Training loop, event history, and readout for the pre-structured policy.

In [ ]:
#| default_exp prestructured_train

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from dataclasses import dataclass
import numpy as np
from explRL.prestructured import angdiff, wrap


In [ ]:
#| export
@dataclass
class EventHistory:
    "Per-event record of one run; headings are radians, times seconds."
    t: np.ndarray          # event start time
    thetaA: np.ndarray     # arena heading before the saccade
    thetaC: np.ndarray     # compass heading before the saccade
    fixDur: np.ndarray     # fixation duration
    direction: np.ndarray  # +1 CW, -1 CCW
    jumped: np.ndarray     # bump jumped after the saccade
    reward: np.ndarray     # Eq. 19 reward at the post-saccade arena heading
    thetaG: np.ndarray     # goal heading after the update


In [ ]:
#| export
def train_prestructured(agent, env, Ttot=None) -> EventHistory:
    "Run events (fixation + saccade) until simulated time reaches Ttot (default params.Ttot)."
    p = agent.params
    Ttot = p.Ttot if Ttot is None else Ttot
    rows, t = [], 0.0
    while t < Ttot:
        ev = agent.rl.chooseAction(agent)
        thetaA1, r = env.step(agent.theta, ev.dx)
        agent.rl.updateParams(agent, ev, r)
        rows.append((t, ev.thetaA, ev.thetaC, ev.fixDur, ev.direction, ev.jumped, r, agent.policy.thetaG))
        agent.theta, agent.thetaC = wrap(thetaA1), ev.thetaC_next
        t += ev.fixDur + p.tS
    cols = list(zip(*rows))
    return EventHistory(*(np.asarray(c) for c in cols[:4]), np.asarray(cols[4], dtype=np.int64),
                        np.asarray(cols[5], dtype=bool), *(np.asarray(c) for c in cols[6:]))


In [ ]:
#| export
def relHeading(thetaA, goal):
    "Arena heading minus goal, in degrees in [-180, 180); positive means clockwise of the goal."
    return np.rad2deg(angdiff(thetaA, goal))

def readout(history, goal, nBins=24) -> dict:
    "Plain-mean fixation duration, turn bias and count per relative-heading bin (nan if empty)."
    rel = relHeading(history.thetaA, goal)
    edges = np.linspace(-180, 180, nBins + 1)
    idx = np.clip(np.digitize(rel, edges) - 1, 0, nBins - 1)
    fix, bias = np.full(nBins, np.nan), np.full(nBins, np.nan)
    count = np.bincount(idx, minlength=nBins)
    for k in np.flatnonzero(count):
        fix[k] = history.fixDur[idx == k].mean()
        bias[k] = history.direction[idx == k].mean()
    return dict(centers=(edges[:-1] + edges[1:]) / 2, fixDur=fix, turnBias=bias, count=count)


In [ ]:
#| export
def heldGoal(history, thetaG0, times, tS):
    "Goal heading in effect at each time: thetaG0 until the first update, then held between updates (each at its event's end)."
    tUpd = history.t + history.fixDur + tS
    idx = np.searchsorted(tUpd, np.asarray(times, dtype=float), side='right') - 1
    return np.where(idx < 0, thetaG0, history.thetaG[np.clip(idx, 0, None)])


In [ ]:
#| export
def runManyPrestructured(params, goal, nRuns, seed=0, Ttot=None, learn=True, thetaG0=None):
    "Independent runs (seed + i); each returns history, readout, starting goal heading and jump rate."
    from explRL.core import loadPrestructuredAgent, loadEnv
    out = []
    for i in range(nRuns):
        agent = loadPrestructuredAgent(params=params, goal=goal, seed=seed + i, thetaG0=thetaG0, learn=learn)
        g0 = agent.policy.thetaG
        hist = train_prestructured(agent, loadEnv(goal=goal), Ttot=Ttot)
        out.append(dict(history=hist, readout=readout(hist, goal), thetaG0=g0, jumpRate=float(hist.jumped.mean())))
    return out


Training loop timing and arrays:

In [ ]:
from math import pi
from fastcore.test import test_eq, test_close
from explRL.core import loadPrestructuredAgent, loadEnv
from explRL.params import Params

p = Params()
agent = loadPrestructuredAgent(params=p, goal=pi, seed=0, learn=False, thetaG0=pi)
env = loadEnv(goal=pi)
hist = train_prestructured(agent, env, Ttot=5.0)
n = len(hist.t)
assert n > 0
for arr in (hist.thetaA, hist.thetaC, hist.fixDur, hist.direction, hist.jumped, hist.reward, hist.thetaG):
    test_eq(len(arr), n)
assert float(hist.t[-1]) < 5.0
assert float(hist.t[-1] + hist.fixDur[-1] + p.tS) >= 5.0
test_close(hist.thetaG, pi)


**AC-7**: Reproducibility and compass offset is always 0 or pi:

In [ ]:
h0 = train_prestructured(loadPrestructuredAgent(seed=1, learn=False, thetaG0=pi), env, Ttot=3.0)
h1 = train_prestructured(loadPrestructuredAgent(seed=1, learn=False, thetaG0=pi), env, Ttot=3.0)
test_close(h0.t, h1.t)
test_close(h0.thetaA, h1.thetaA)
h2 = train_prestructured(loadPrestructuredAgent(seed=2, learn=False, thetaG0=pi), env, Ttot=3.0)
assert len(h2.t) != len(h0.t) or not np.allclose(h2.thetaA, h0.thetaA)
off = np.abs(angdiff(h0.thetaC, h0.thetaA))
assert np.all(np.isclose(off, 0.0) | np.isclose(off, pi))


`relHeading` and `readout` on a hand-built history:

In [ ]:
test_close(relHeading(np.array([pi, pi + 0.1]), pi)[1], np.rad2deg(0.1))
hand = EventHistory(
    t=np.array([0.0, 1.0]),
    thetaA=np.array([pi, 0.0]),
    thetaC=np.array([pi, pi]),
    fixDur=np.array([0.5, 1.5]),
    direction=np.array([1, -1], dtype=np.int64),
    jumped=np.array([False, True]),
    reward=np.array([0.1, -0.1]),
    thetaG=np.array([pi, pi]),
)
ro = readout(hand, pi, nBins=2)
test_eq(ro['count'].sum(), 2)
test_close(ro['fixDur'][1], 0.5)   # rel = 0 at the goal
test_close(ro['turnBias'][1], 1.0)
test_close(ro['fixDur'][0], 1.5)   # rel = -180 at the anti-goal
test_close(ro['turnBias'][0], -1.0)


In [ ]:
ro4 = readout(hand, pi, nBins=4)
test_eq(ro4['count'].tolist(), [1, 0, 1, 0])
assert np.isnan(ro4['fixDur'][1]) and np.isnan(ro4['turnBias'][3])
test_close(relHeading(0.0, pi), -180.0)
assert np.all(relHeading(np.linspace(0, 2 * pi, 50), 1.0) < 180)


In [ ]:
# hand history: updates land at 0 + 0.5 + tS and 1 + 1.5 + tS
tS = 0.32
test_close(heldGoal(hand, 2.0, [0.0, 0.81, 2.81, 100.0], tS), [2.0, 2.0, pi, pi])
hand2 = EventHistory(**{**hand.__dict__, 'thetaG': np.array([1.0, 3.0])})
test_close(heldGoal(hand2, 2.0, [0.5, 0.83, 2.81, 2.83], tS), [2.0, 1.0, 1.0, 3.0])


In [ ]:
runs = runManyPrestructured(params=p, goal=pi, nRuns=2, seed=10, Ttot=30.0, learn=False, thetaG0=pi)
test_eq(len(runs), 2)
for d in runs:
    test_eq(set(d.keys()), {'history', 'readout', 'thetaG0', 'jumpRate'})
    test_close(d['thetaG0'], pi)


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()